# Module 07 — Notebook 4 Solutions: Mini-Project

In [ ]:
import sys
sys.path.insert(0, "../../../")
from src.checks import check_equal, check_type, check_contains
import subprocess
import json
import random
import numpy as np
import importlib.metadata
from pathlib import Path

SCRIPTS_DIR = Path("scripts")
OUTPUT_DIR = Path("output")
SCRIPTS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
print("Setup complete.")

## Step 1 — Solution: The Analysis Script

In [ ]:
%%writefile scripts/analyze_scores.py
"""
analyze_scores.py — Reproducible model output analysis.

Usage:
    python scripts/analyze_scores.py --input ../../data/synthetic/model_outputs.json
"""
import argparse
import json
import logging
import random
from pathlib import Path

import numpy as np

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger(__name__)


def load_outputs(path):
    """Load a JSON file of model outputs."""
    with open(path) as f:
        return json.load(f)


def compute_stats(outputs):
    """Compute mean, min, max of scores across all outputs."""
    scores = [o["score"] for o in outputs]
    return {
        "n": len(scores),
        "mean": round(sum(scores) / len(scores), 4),
        "min": round(min(scores), 4),
        "max": round(max(scores), 4),
    }


def sample_outputs(outputs, n=5, seed=42):
    """Randomly sample n outputs using a local RNG to avoid affecting global state."""
    rng = random.Random(seed)
    return rng.sample(outputs, min(n, len(outputs)))


def main():
    parser = argparse.ArgumentParser(description="Analyze model output scores.")
    parser.add_argument("--input", type=Path, required=True, help="Path to model_outputs.json")
    parser.add_argument("--output", type=Path, default=Path("output/stats.json"), help="Where to save results")
    args = parser.parse_args()

    outputs = load_outputs(args.input)
    log.info(f"Loaded {len(outputs)} outputs from {args.input}")

    stats = compute_stats(outputs)
    log.info(f"Stats: {stats}")

    sample = sample_outputs(outputs, n=5, seed=SEED)

    result = {"stats": stats, "sample": sample}

    args.output.parent.mkdir(parents=True, exist_ok=True)
    with open(args.output, "w") as f:
        json.dump(result, f, indent=2)

    log.info(f"Saved results to {args.output}")


if __name__ == "__main__":
    main()

In [ ]:
script_path = Path("scripts/analyze_scores.py")
check_equal(script_path.exists(), True, "scripts/analyze_scores.py exists")

source = script_path.read_text()
check_contains(source, "SEED", "SEED constant is defined")
check_contains(source, "random.seed", "random.seed is called")
check_contains(source, "np.random.seed", "np.random.seed is called")
check_contains(source, "def main", "main() function is defined")
check_contains(source, "__name__", "entry-point guard exists")
check_contains(source, "argparse", "argparse is used")
check_contains(source, "--input", "--input argument is defined")

## Step 2 — Solution: Verify Reproducibility

In [ ]:
DATA = Path("../../../data/synthetic/model_outputs.json")
OUTPUT_1 = Path("output/run1_stats.json")
OUTPUT_2 = Path("output/run2_stats.json")

result1 = subprocess.run(
    [sys.executable, "scripts/analyze_scores.py", "--input", str(DATA), "--output", str(OUTPUT_1)],
    capture_output=True, text=True
)
result2 = subprocess.run(
    [sys.executable, "scripts/analyze_scores.py", "--input", str(DATA), "--output", str(OUTPUT_2)],
    capture_output=True, text=True
)

print("Run 1 exit code:", result1.returncode)
print("Run 2 exit code:", result2.returncode)

In [ ]:
with open(OUTPUT_1) as f:
    run1 = json.load(f)

with open(OUTPUT_2) as f:
    run2 = json.load(f)

runs_are_identical = (run1 == run2)

In [ ]:
check_equal(result1.returncode, 0, "run 1 exited successfully")
check_equal(result2.returncode, 0, "run 2 exited successfully")
check_equal(runs_are_identical, True, "both runs produce identical output (reproducibility verified!)")

## Step 3 — Solution: requirements.txt

In [ ]:
import importlib.metadata
for pkg in ["numpy", "pandas", "matplotlib"]:
    try:
        v = importlib.metadata.version(pkg)
        print(f"{pkg}=={v}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg} — not installed")

In [ ]:
%%writefile requirements.txt
# Core analysis dependencies
numpy==1.26.4
pandas==2.2.0
matplotlib==3.8.3

In [ ]:
req_path = Path("requirements.txt")
check_equal(req_path.exists(), True, "requirements.txt exists")

content = req_path.read_text()
check_contains(content, "numpy==", "numpy is pinned with ==")
check_contains(content, "pandas==", "pandas is pinned with ==")

pinned_lines = [l for l in content.splitlines() if "==" in l]
check_equal(len(pinned_lines) >= 2, True, "at least 2 packages are pinned")

## Step 4 — Solution: REPRODUCE.md

In [ ]:
%%writefile REPRODUCE.md
# Reproduction Guide

Follow these steps to reproduce the analysis from scratch.

## Setup

Create and activate a virtual environment:

```bash
uv venv .venv
source .venv/bin/activate   # Mac/Linux
# .venv\Scripts\activate    # Windows
```

## Install Dependencies

Install the exact package versions used in this analysis:

```bash
uv pip install -r requirements.txt
```

## Run Analysis

From the `notebooks/module_07_environments/` directory:

```bash
python scripts/analyze_scores.py --input ../../data/synthetic/model_outputs.json
```

Output is saved to `output/stats.json`. Running this command twice should produce identical output files.

In [ ]:
md_path = Path("REPRODUCE.md")
check_equal(md_path.exists(), True, "REPRODUCE.md exists")

content = md_path.read_text()
check_contains(content, "## Setup", "REPRODUCE.md has a ## Setup section")
check_contains(content, "requirements.txt", "REPRODUCE.md references requirements.txt")
check_contains(content, "analyze_scores.py", "REPRODUCE.md references the analysis script")